<a href="https://colab.research.google.com/github/shivamagarwalhere/practice-repository/blob/main/RAG_QA_Gemini_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG QA System for AI Research Papers
This notebook builds a Retrieval-Augmented Generation (RAG) system for question answering over a local collection of AI research papers using LangChain and Gemini.

The workflow includes document loading, text chunking, vector embedding, retrieval with FAISS, Gemini answer generation, and source attribution.

In [11]:
# Install required packages with updated LangChain v1.x modular packages
import sys

!{sys.executable} -m pip install -U --quiet langchain langchain-classic langchain-community langchain-google-genai langchain-text-splitters unstructured[local] faiss-cpu pypdf markdown

In [12]:
import os
from pathlib import Path
from getpass import getpass

import markdown as md
from IPython.display import Markdown, display

# 1. Document Loaders & Vector Stores (Community Package)
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

# 2. Text Splitters (Dedicated Package)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 3. Google GenAI Integrations (Partner Package)
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# 4. Chains & Core Prompts (Classic & Core Packages)
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [13]:
workspace = Path.cwd()
pdf_files = sorted(workspace.glob("*.pdf"))
assert pdf_files, "No PDF files found in the notebook workspace. Please place the papers in the same folder."

display(Markdown(f"### Found {len(pdf_files)} PDFs for processing:"))
for pdf_path in pdf_files:
    display(Markdown(f"- **{pdf_path.name}**"))

source_documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    display(Markdown(f"Loaded **{len(pages)}** pages from **{pdf_path.name}**."))
    source_documents.extend(pages)

assert source_documents, "No document pages were loaded."

### Found 3 PDFs for processing:

- **1706.03762v7.pdf**

- **2005.11401v4.pdf**

- **2005.14165v4.pdf**

Loaded **15** pages from **1706.03762v7.pdf**.

Loaded **19** pages from **2005.11401v4.pdf**.

Loaded **75** pages from **2005.14165v4.pdf**.

In [14]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(source_documents)

display(Markdown(f"### Split documents into **{len(chunks)}** overlapping chunks."))

for idx, chunk in enumerate(chunks[:3], start=1):
    preview = chunk.page_content.replace("\n", " ")[:220]
    display(Markdown(f"**Chunk {idx} preview:** {preview}..."))

### Split documents into **461** overlapping chunks.

**Chunk 1 preview:** Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Goog...

**Chunk 2 preview:** mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show thes...

**Chunk 3 preview:** best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data. ∗Equal contribution. Li...

In [15]:
gemini_api_key = os.getenv("GEMINI_API_KEY") or getpass("Enter Gemini API key: ")
os.environ["GOOGLE_API_KEY"] = gemini_api_key

Enter Gemini API key: ··········


In [16]:
import google.generativeai as genai

# Configure with the API key
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Listing all available Gemini models:")
for m in genai.list_models():
    # Filter to show only models that support 'embedContent'
    if "embedContent" in m.supported_generation_methods:
        print(f"  Model name: {m.name}, Description: {m.description}")

Listing all available Gemini models:
  Model name: models/gemini-embedding-001, Description: Obtain a distributed representation of a text.
  Model name: models/gemini-embedding-2-preview, Description: Obtain a distributed representation of multimodal content.
  Model name: models/gemini-embedding-2, Description: Obtain a distributed representation of multimodal content.


In [19]:
import time
from math import ceil

# Using Google's updated embedding model. The `batch_size` parameter was previously removed,
# and re-adding it did not resolve the underlying issue with `embed_documents`.
# We will now manually manage embedding generation for each document.
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")

# Preprocess chunks to handle UnicodeEncodeError by ensuring UTF-8 compatibility
cleaned_chunks = []
for doc in chunks:
    cleaned_content = doc.page_content.encode('utf-8', errors='replace').decode('utf-8')
    doc.page_content = cleaned_content
    cleaned_chunks.append(doc)

vectorstore = None
batch_size_for_faiss = 25 # As requested by user, 25 documents per actual embedding call

num_batches = ceil(len(cleaned_chunks) / batch_size_for_faiss)

display(Markdown(f"### Embedding {len(cleaned_chunks)} chunks in {num_batches} batches with a 1-minute delay between batches."))

for i in range(num_batches):
    start_idx = i * batch_size_for_faiss
    end_idx = min((i + 1) * batch_size_for_faiss, len(cleaned_chunks))
    current_batch = cleaned_chunks[start_idx:end_idx]

    display(Markdown(f"Processing batch {i+1}/{num_batches} ({len(current_batch)} documents)..."))

    # Manually generate embeddings for each document in the batch using embed_query
    batch_texts = [doc.page_content for doc in current_batch]
    batch_metadatas = [doc.metadata for doc in current_batch]
    batch_individual_embeddings = []

    # Use embed_query for each text to ensure one embedding per document
    for text in batch_texts:
        batch_individual_embeddings.append(embeddings.embed_query(text))

    # Prepare text_embeddings for FAISS.from_embeddings/add_embeddings
    text_embeddings_tuples = list(zip(batch_texts, batch_individual_embeddings))

    if vectorstore is None:
        # For the first batch, initialize the vectorstore using pre-computed embeddings
        vectorstore = FAISS.from_embeddings(text_embeddings_tuples, embeddings, metadatas=batch_metadatas)
    else:
        # For subsequent batches, add documents to the existing vectorstore using pre-computed embeddings
        vectorstore.add_embeddings(text_embeddings_tuples, metadatas=batch_metadatas)

    if i < num_batches - 1: # Don't sleep after the very last batch
        display(Markdown("Sleeping for 1 minute before the next batch to respect API rate limits..."))
        time.sleep(60) # 1-minute delay

display(Markdown("### Embeddings created and FAISS vector store initialized."))

### Embedding 461 chunks in 19 batches with a 1-minute delay between batches.

Processing batch 1/19 (25 documents)...

Sleeping for 1 minute before the next batch to respect API rate limits...

Processing batch 2/19 (25 documents)...

Sleeping for 1 minute before the next batch to respect API rate limits...

...

Sleeping for 1 minute before the next batch to respect API rate limits...

Processing batch 18/19 (25 documents)...

Sleeping for 1 minute before the next batch to respect API rate limits...

Processing batch 19/19 (11 documents)...

### Embeddings created and FAISS vector store initialized.

In [20]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

display(Markdown("### Retriever built and tuned for top-4 results."))

### Retriever built and tuned for top-4 results.

In [25]:
os.environ["GOOGLE_API_KEY"] = gemini_api_key

# Using the modern 1.5 Flash model
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash-lite", temperature=0.2, max_output_tokens=512)

display(Markdown("### Gemini LLM configured for answer generation."))

### Gemini LLM configured for answer generation.

In [26]:
# 1. Define a system prompt for the AI to follow
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

# 2. Build the LCEL Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Create the modern chains
question_answer_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

display(Markdown("### Modern RAG QA chain assembled using LCEL."))

sample_questions = [
    "What are the main components of a RAG model, and how do they interact?",
    "What are the two sub-layers in each encoder layer of the Transformer model?",
    "Explain how positional encoding is implemented in Transformers and why it is necessary.",
    "Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?",
    "What is few-shot learning, and how does GPT-3 implement it during inference?",
]

def format_sources(source_docs):
    pieces = []
    for idx, source_doc in enumerate(source_docs, start=1):
        source_name = Path(source_doc.metadata.get("source", "")).name or "unknown source"
        page = source_doc.metadata.get("page", "N/A")
        preview = source_doc.page_content.replace("\n", " ").strip()[:300]
        pieces.append(
            f"**Source {idx}:** {source_name} | page {page}\n\n> {preview}..."
        )
    return "\n\n".join(pieces)

def ask_and_print(question: str):
    # Invoke using modern LCEL syntax
    result = qa_chain.invoke({"input": question})

    # Extract answer and context based on LCEL dict structure
    answer_text = result.get("answer", "")
    source_docs = result.get("context", [])

    markdown_text = (
        f"## Question\n{question}\n\n"
        f"## Answer\n{answer_text}\n\n"
        f"## Source Attribution\n{format_sources(source_docs)}"
    )
    html_text = md.markdown(markdown_text)
    display(Markdown(markdown_text))
    print("\n--- Raw HTML generated from python markdown library ---\n")
    print(html_text)

for question in sample_questions:
    ask_and_print(question)

### Modern RAG QA chain assembled using LCEL.

## Question
What are the main components of a RAG model, and how do they interact?

## Answer
A RAG model consists of two main components: a retriever and a generator. The retriever, using a pre-trained neural network, accesses a dense vector index (like Wikipedia) to find relevant documents based on the input query. The generator, a pre-trained seq2seq model, then uses both the original input and the retrieved documents to produce the output. These components are trained end-to-end in a probabilistic model.

## Source Attribution
**Source 1:** 2005.11401v4.pdf | page 1

> non-parametric memory is a dense vector index of Wikipedia, accessed with a pre-trained neural retriever. We combine these components in a probabilistic model trained end-to-end (Fig. 1). The retriever (Dense Passage Retriever [26], henceforth DPR) provides latent documents conditioned on the input,...

**Source 2:** 2005.11401v4.pdf | page 1

> Index) with a pre-trained seq2seq model (Generator) and ﬁne-tune end-to-end. For queryx, we use Maximum Inner Product Search (MIPS) to ﬁnd the top-K documentszi. For ﬁnal predictiony, we treatz as a latent variable and marginalize over seq2seq predictions given different documents. but have only exp...

**Source 3:** 2005.11401v4.pdf | page 1

> 2 Methods We explore RAG models, which use the input sequencex to retrieve text documentsz and use them as additional context when generating the target sequence y. As shown in Figure 1, our models leverage two components: (i) a retriever pη(z|x) with parametersη that returns (top-K truncated) distr...

**Source 4:** 2005.11401v4.pdf | page 0

> per token. We ﬁne-tune and evaluate our models on a wide range of knowledge- intensive NLP tasks and set the state of the art on three open domain QA tasks, outperforming parametric seq2seq models and task-speciﬁc retrieve-and-extract architectures. For language generation tasks, we ﬁnd that RAG mod...


--- Raw HTML generated from python markdown library ---

<h2>Question</h2>
<p>What are the main components of a RAG model, and how do they interact?</p>
<h2>Answer</h2>
<p>A RAG model consists of two main components: a retriever and a generator. The retriever, using a pre-trained neural network, accesses a dense vector index (like Wikipedia) to find relevant documents based on the input query. The generator, a pre-trained seq2seq model, then uses both the original input and the retrieved documents to produce the output. These components are trained end-to-end in a probabilistic model.</p>
<h2>Source Attribution</h2>
<p><strong>Source 1:</strong> 2005.11401v4.pdf | page 1</p>
<blockquote>
<p>non-parametric memory is a dense vector index of Wikipedia, accessed with a pre-trained neural retriever. We combine these components in a probabilistic model trained end-to-end (Fig. 1). The retriever (Dense Passage Retriever [26], henceforth DPR) provides latent documents conditioned on the input

## Question
What are the two sub-layers in each encoder layer of the Transformer model?

## Answer
Each encoder layer in the Transformer model consists of two sub-layers. The first is a multi-head self-attention mechanism, and the second is a position-wise fully connected feed-forward network.

## Source Attribution
**Source 1:** 1706.03762v7.pdf | page 2

> Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder...

**Source 2:** 1706.03762v7.pdf | page 4

> encoder. • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of...

**Source 3:** 1706.03762v7.pdf | page 2

> itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder ...

**Source 4:** 1706.03762v7.pdf | page 0

> best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data. ∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and sta...


--- Raw HTML generated from python markdown library ---

<h2>Question</h2>
<p>What are the two sub-layers in each encoder layer of the Transformer model?</p>
<h2>Answer</h2>
<p>Each encoder layer in the Transformer model consists of two sub-layers. The first is a multi-head self-attention mechanism, and the second is a position-wise fully connected feed-forward network.</p>
<h2>Source Attribution</h2>
<p><strong>Source 1:</strong> 1706.03762v7.pdf | page 2</p>
<blockquote>
<p>Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder...</p>
</blockquote>
<p><strong>Source 2:</strong> 1706.03762v7.pdf | page 4</p>
<blockquote>
<p>encoder. • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions

...


--- Raw HTML generated from python markdown library ---

<h2>Question</h2>
<p>Explain how positional encoding is implemented in Transformers and why it is necessary.</p>
<h2>Answer</h2>
<p>Positional encoding is necessary in Transformers because the model architecture, lacking recurrence or convolution, needs explicit information about the order of tokens in a sequence. It is implemented by adding "positional encodings" to the input embeddings. These encodings use sine and cosine functions of different frequencies, allowing the model to easily learn to attend by relative positions.</p>
<h2>Source Attribution</h2>
<p><strong>Source 1:</strong> 1706.03762v7.pdf | page 5</p>
<blockquote>
<p>tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel as the embeddings, so that the two can be summed. There are many choices of positional encodings, learn..

## Question
Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?

## Answer
Multi-head attention allows the Transformer model to jointly attend to information from different representation subspaces at different positions. It achieves this by running multiple attention mechanisms in parallel, each with its own learned projections. This is beneficial because it enables the model to capture a richer set of dependencies and relationships within the data compared to a single attention mechanism.

## Source Attribution
**Source 1:** 1706.03762v7.pdf | page 4

> output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhib...

**Source 2:** 1706.03762v7.pdf | page 4

> The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in ...

**Source 3:** 1706.03762v7.pdf | page 0

> mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring...

**Source 4:** 1706.03762v7.pdf | page 1

> in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due t...


--- Raw HTML generated from python markdown library ---

<h2>Question</h2>
<p>Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?</p>
<h2>Answer</h2>
<p>Multi-head attention allows the Transformer model to jointly attend to information from different representation subspaces at different positions. It achieves this by running multiple attention mechanisms in parallel, each with its own learned projections. This is beneficial because it enables the model to capture a richer set of dependencies and relationships within the data compared to a single attention mechanism.</p>
<h2>Source Attribution</h2>
<p><strong>Source 1:</strong> 1706.03762v7.pdf | page 4</p>
<blockquote>
<p>output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single 

## Question
What is few-shot learning, and how does GPT-3 implement it during inference?

## Answer
Few-shot learning (FS) is a setting where a model is given a few demonstrations of a task at inference time as conditioning, without any weight updates. GPT-3 implements this by providing the model with a few examples of context and completion, followed by a final context, and then expecting the model to generate the completion. This approach significantly reduces the need for task-specific data.

## Source Attribution
**Source 1:** 2005.14165v4.pdf | page 5

> this work we do not ﬁne-tune GPT-3 because our focus is on task-agnostic performance, but GPT-3 can be ﬁne-tuned in principle and this is a promising direction for future work. • Few-Shot (FS) is the term we will use in this work to refer to the setting where the model is given a few demonstrations ...

**Source 2:** 2005.14165v4.pdf | page 33

> information, or from algorithmic improvements. A limitation, or at least uncertainty, associated with few-shot learning in GPT-3 is ambiguity about whether few-shot learning actually learns new tasks “from scratch” at inference time, or if it simply recognizes and identiﬁes tasks that it has learned...

**Source 3:** 2005.14165v4.pdf | page 4

> number of examples in-context hold for most tasks we study. We emphasize that these “learning” curves involve no gradient updates or ﬁne-tuning, just increasing numbers of demonstrations given as conditioning. Broadly, on NLP tasks GPT-3 achieves promising results in the zero-shot and one-shot setti...

**Source 4:** 2005.14165v4.pdf | page 0

> thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions – something which current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performa...


--- Raw HTML generated from python markdown library ---

<h2>Question</h2>
<p>What is few-shot learning, and how does GPT-3 implement it during inference?</p>
<h2>Answer</h2>
<p>Few-shot learning (FS) is a setting where a model is given a few demonstrations of a task at inference time as conditioning, without any weight updates. GPT-3 implements this by providing the model with a few examples of context and completion, followed by a final context, and then expecting the model to generate the completion. This approach significantly reduces the need for task-specific data.</p>
<h2>Source Attribution</h2>
<p><strong>Source 1:</strong> 2005.14165v4.pdf | page 5</p>
<blockquote>
<p>this work we do not ﬁne-tune GPT-3 because our focus is on task-agnostic performance, but GPT-3 can be ﬁne-tuned in principle and this is a promising direction for future work. • Few-Shot (FS) is the term we will use in this work to refer to the setting where the model is given a few demonstrations ...</p>
</blo